# Uninformed Search
*Important:* Remember to run cells to see the results. Some cells rely on cells further up the page being run, so be aware of this as you make modifications. Run each cell as you work your way down the page even if results are already showing. If something isn't working, restart the kernel, and rerun the entire notebook.

## Tower of Hanoi
### Introduction

The Tower of Hanoi is a well known mathematical puzzle from 1883. You are given three pegs which hold a number of circular disks of different sizes. The disks all start on the furthest left peg, and you must move the entire tower to the furthest right peg, one disk at a time, obeying a simple rule: you may not put a disk on top of a smaller disk.

<img src="https://upload.wikimedia.org/wikipedia/commons/0/07/Tower_of_Hanoi.jpeg" align=center />

<p style="text-align: center;">Tower of Hanoi puzzle with 8 disks in its start configuration. Image from <a href="https://commons.wikimedia.org/w/index.php?curid=228623">Wikimedia</a> under <a href="http://creativecommons.org/licenses/by-sa/3.0/" title="Creative Commons Attribution-Share Alike 3.0">CC BY-SA 3.0</a> licence.</p>

### Code
There is a Python module in this project which implements the Tower of Hanoi game called `hanoi.py`. You do not need to look at the code – you can if you wish, but I suggest working through this entire notebook once first. It contains a class called `HanoiState` which allows us to represent the state of the puzzle. When you create a new state with no additional arguments, it will create the starting state for a 5-disk puzzle:

In [ ]:
from hanoi import HanoiState

state = HanoiState()
print(state)

Notice that numbers are used to represent the size of the disks, with `1` being the smallest. We can provide an argument to the constructor to create a state with a different number of disks (or pegs):

In [ ]:
state = HanoiState(disks=3)
print(state)

From any state, we can call the `possible_actions` method to find what actions are available:

In [ ]:
print(state.possible_actions())

The `possible_actions` method returns a list of tuples. A single tuple `(x, y)` indicates that it is possible to move a disk from peg `x` to peg `y`, with the first peg being numbered `0`.

We can use this information, or otherwise, to move disks around. We do this using the `next_state(from_peg, to_peg)` method.

In [ ]:
state = state.next_state(0, 2)
print(state)

*Important point:* note that HanoiState objects are *immutable*. You cannot change their internal state. We had to write:
```python
state = state.next_state(0, 2)
```

rather than
```python
state.next_state(0, 2)
```

Simply calling the method and ignoring the result will not change the original object:

In [ ]:
test = HanoiState(disks=3)
test.next_state(0, 2)
print(test)

This is a really useful property when writing search algorithms, because it means we can keep track of which states we've seen before, and we don't have to worry about accidentally changing their internal values later. This property is explained in more detail in [a separate notebook](additional/immutable_state.ipynb), but it will be helpful to understand the basics of the breadth first search algorithm first.

Back to the code. There is a method which we can call to query whether the state is a goal state: i.e. has the game been solved, are all the disks on the furthest right peg?

In [ ]:
print(state)
print(state.is_goal_state())

Now let's do the full series of moves required to solve the puzzle with 3 disks:

In [ ]:
state = state.next_state(0, 1)
state = state.next_state(2, 1)
state = state.next_state(0, 2)
state = state.next_state(1, 0)
state = state.next_state(1, 2)
state = state.next_state(0, 2)
print(state)
print(state.is_goal_state())

### Breadth-First Search
Given the starting configuration with all of the disks on the first peg, we'd like to search for the final configuration, with all disks on the rightmost peg. Our real objective here is to find the *path* that actually takes us from one to the other, but as described in the videos, this is something we will easily be able to solve by working backwards once we have found it.

Here is a copy of the breadth-first search algorithm, pseudocode taken from Russel and Norvig (p. 82):<br />
<img src="./additional/Breadth_first_search.png" width=60% /> <br />

And here is an implementation for the Tower of Hanoi problem:

In [13]:
def breadth_first_search():
    # 3 disks keeps the printed output short enough to read – change back to 5 to see the full search
    state = HanoiState(disks=3)
    frontier = [state]
    explored = []
    iteration = 0

    print(f"Initial frontier: {frontier}")

    current_state = frontier.pop(0)
    while not current_state.is_goal_state():
        iteration += 1
        print(f"\n--- Iteration {iteration} ---")
        print(f"Expanding: {current_state!r}")

        explored.append(current_state)
        actions = current_state.possible_actions()
        for action in actions:
            # remember next_state(from peg, to peg)
            new_state = current_state.next_state(action[0], action[1])
            if new_state not in explored and new_state not in frontier:
                frontier.append(new_state)
                print(f"  action {action} -> added {new_state!r}")
            else:
                print(f"  action {action} -> skipped {new_state!r} (already seen)")

        print(f"Frontier now ({len(frontier)} states): {frontier}")
        print(f"Explored so far: {len(explored)} states")

        if len(frontier) == 0:
            return None

        current_state = frontier.pop(0)

    print(f"\nGoal reached after {iteration} iterations: {current_state!r}")
    return current_state


final_state = breadth_first_search()

if final_state is None:
    print("No solution found...")
else:
    print("Solution found!")
    print(final_state)

Initial frontier: [((3, 2, 1), (), ())]

--- Iteration 1 ---
Expanding: ((3, 2, 1), (), ())
  action (0, 1) -> added ((3, 2), (1,), ())
  action (0, 2) -> added ((3, 2), (), (1,))
Frontier now (2 states): [((3, 2), (1,), ()), ((3, 2), (), (1,))]
Explored so far: 1 states

--- Iteration 2 ---
Expanding: ((3, 2), (1,), ())
  action (0, 2) -> added ((3,), (1,), (2,))
  action (1, 0) -> skipped ((3, 2, 1), (), ()) (already seen)
  action (1, 2) -> skipped ((3, 2), (), (1,)) (already seen)
Frontier now (2 states): [((3, 2), (), (1,)), ((3,), (1,), (2,))]
Explored so far: 2 states

--- Iteration 3 ---
Expanding: ((3, 2), (), (1,))
  action (0, 1) -> added ((3,), (2,), (1,))
  action (2, 0) -> skipped ((3, 2, 1), (), ()) (already seen)
  action (2, 1) -> skipped ((3, 2), (1,), ()) (already seen)
Frontier now (2 states): [((3,), (1,), (2,)), ((3,), (2,), (1,))]
Explored so far: 3 states

--- Iteration 4 ---
Expanding: ((3,), (1,), (2,))
  action (1, 0) -> added ((3, 1), (), (2,))
  action (1, 

So we know the goal state is found but how do we then get there?

Cost of planning we know is 24 iterations but how much depth is needed to reach the goal?

Make sure you spend a good amount of time with this code. Remember you can modify it and re-run it to see the results. You could add print statements which show the contents of the frontier at each point in the algorithm, for example.

### Finding The Path
We can make a small modification to this algorithm which will allow us to work backwards from the goal state to the start state, thereby showing us the entire sequence of moves which solves the game.

`HanoiState` objects have an attribute called `parent`. When you create a state using the `next_state` method, the attribute of the new state is set automatically to point at the original state. Here is a demonstration:

In [14]:
state = HanoiState(disks=3)
state = state.next_state(0, 2)
print("Here is the value of state:")
print(state)
print("Here is the value of state.parent")
print(state.parent)
print("Here is the value of state.parent.parent")
print(state.parent.parent)

Here is the value of state:
 | | |
 2 | |
 3 | 1

Here is the value of state.parent
 1 | |
 2 | |
 3 | |

Here is the value of state.parent.parent
None


So each state has a reference to its parent state, all the way back to the start state, whose `parent` attribute is set to `None`. This means that once we've found the goal state, we can easily backtrack by following the chain with something like a while loop. 

Here is a new version of `breadth_first_search()` which prints the entire path once the solution has been found. 
* Read the code carefully to see how it iterates through the states backwards, adding them to a list, and then prints this list backwards to obtain the result in the correct order. 
* Notice that the code also keeps track of some simple metrics that allow us to compare how efficiently the algorithm is performing: it will print the total number of states that it generates, and the total number it explores (the number of unique states it generates).

In [15]:
def breadth_first_search():
    state = HanoiState(disks=5)
    frontier = [state]
    explored = []
    generated = 0

    current_state = frontier.pop(0)
    while not current_state.is_goal_state():
        explored.append(current_state)
        actions = current_state.possible_actions()
        for action in actions:
            generated += 1
            new_state = current_state.next_state(action[0], action[1])
            if new_state not in explored and new_state not in frontier:
                frontier.append(new_state)

        if len(frontier) == 0:
            print("No solution found")
            return

        current_state = frontier.pop(0)

    print("Solution found!")
    print(f"Explored {len(explored)} states")
    print(f"Generated {generated} states")
    print()

    # NOTE: final_path is built BACKWARDS (goal -> start).
    # We only have .parent pointers (child -> parent), there is no .child pointer,
    # so the only direction we can walk is from the goal back towards the start.
    # After this loop: final_path == [goal, ..., start]
    final_path = []
    while current_state.parent is not None:
        final_path.append(current_state)
        current_state = current_state.parent

    # The loop stops when it reaches the start state (parent is None), but that
    # state has not been appended yet, so add it here.
    final_path.append(current_state)

    # reversed() reads the list from the end, so this prints start -> goal,
    # i.e. the order you would actually play the moves in.
    for state in reversed(final_path):
        # NOTE: state.action is the single (from_peg, to_peg) move that LED TO this
        # state - not to be confused with possible_actions(), which is the list of
        # moves available FROM it. Only the start state has action == None, because
        # it was created directly with HanoiState() rather than by a move, so this
        # check just skips the "Move disk..." line for the very first state.
        if state.action is not None:
            print(f"Move disk from peg {state.action[0]} to {state.action[1]}")
        print(state)
    #a "step" is a move between two states, so there's always one fewer move than there are states.
    print(f"Total {len(final_path)-1} steps")

breadth_first_search()

Solution found!
Explored 232 states
Generated 694 states

 1 | |
 2 | |
 3 | |
 4 | |
 5 | |

Move disk from peg 0 to 2
 | | |
 2 | |
 3 | |
 4 | |
 5 | 1

Move disk from peg 0 to 1
 | | |
 | | |
 3 | |
 4 | |
 5 2 1

Move disk from peg 2 to 1
 | | |
 | | |
 3 | |
 4 1 |
 5 2 |

Move disk from peg 0 to 2
 | | |
 | | |
 | | |
 4 1 |
 5 2 3

Move disk from peg 1 to 0
 | | |
 | | |
 1 | |
 4 | |
 5 2 3

Move disk from peg 1 to 2
 | | |
 | | |
 1 | |
 4 | 2
 5 | 3

Move disk from peg 0 to 2
 | | |
 | | |
 | | 1
 4 | 2
 5 | 3

Move disk from peg 0 to 1
 | | |
 | | |
 | | 1
 | | 2
 5 4 3

Move disk from peg 2 to 1
 | | |
 | | |
 | | |
 | 1 2
 5 4 3

Move disk from peg 2 to 0
 | | |
 | | |
 | | |
 2 1 |
 5 4 3

Move disk from peg 1 to 0
 | | |
 | | |
 1 | |
 2 | |
 5 4 3

Move disk from peg 2 to 1
 | | |
 | | |
 1 | |
 2 3 |
 5 4 |

Move disk from peg 0 to 2
 | | |
 | | |
 | | |
 2 3 |
 5 4 1

Move disk from peg 0 to 1
 | | |
 | | |
 | 2 |
 | 3 |
 5 4 1

Move disk from peg 2 to 1
 | | |
 | 1 

So:

- explored = 24 → how expensive the planning was (time and memory)
- steps = 7 → how good the plan is

For the Tower of Hanoi, it is known that for $n$ disks, the shortest path from the start state to the end state is $2^n-1$ moves (a total of $2^n$ states including the start state). As the result at the end of the code shows, our breadth first search found a sequence of moves which was optimal. (Notice that `len(final_path)` is the total number of states including the start state, so `len(final_path)-1` is the number of moves.)

### Your Turn
If you haven't already, try modifying the code above to produce different results. What happens if you change the number of disks, or even the number of pegs? The search space can get large quite quickly, so don't forget you can press the stop button on the toolbar to terminate a cell that has been running for too long.

Breadth first search is guaranteed to find a solution with the minimum number of steps, because it checks all solutions of length $n$ before it starts trying solutions of length $n+1$. However, storing all of the unexplored states (the frontier) uses a lot of memory.

We also learned about *depth first search*. In a breadth first search, the shallowest node in the frontier is expanded first, whereas in a depth first search, it is the deepest.

**Exercise:** Write your own code in the cell below to perform depth first search. It can be done with a surprisingly small modification to the breadth first graph search code. This is described in more detail in the textbook. It can also be implemented with a recursive function, but this will likely be more work. 

Answer the following questions of your depth first search solution:
* Is the result of optimal length (in number of steps)? 
* Does it generate more or fewer states, using the same metrics as the previous code? 
* When do you think we'd be likely to use depth first search, and when would we use breadth first?
* If this version is not optimal, what modifications would be necessary to find an optimal result? 

In [17]:
def depth_first_search():
    state = HanoiState(disks=5)
    frontier = [state]
    explored = []
    generated = 0

    current_state = frontier.pop(-1)
    while not current_state.is_goal_state():
        explored.append(current_state)
        actions = current_state.possible_actions()
        for action in actions:
            generated += 1
            new_state = current_state.next_state(action[0], action[1])
            if new_state not in explored and new_state not in frontier:
                frontier.append(new_state)

        if len(frontier) == 0:
            print("No solution found")
            return

        current_state = frontier.pop(-1)

    print("Solution found!")
    print(f"Explored {len(explored)} states")
    print(f"Generated {generated} states")
    print()

    # NOTE: final_path is built BACKWARDS (goal -> start).
    # We only have .parent pointers (child -> parent), there is no .child pointer,
    # so the only direction we can walk is from the goal back towards the start.
    # After this loop: final_path == [goal, ..., start]
    final_path = []
    while current_state.parent is not None:
        final_path.append(current_state)
        current_state = current_state.parent

    # The loop stops when it reaches the start state (parent is None), but that
    # state has not been appended yet, so add it here.
    final_path.append(current_state)

    # reversed() reads the list from the end, so this prints start -> goal,
    # i.e. the order you would actually play the moves in.
    for state in reversed(final_path):
        # NOTE: state.action is the single (from_peg, to_peg) move that LED TO this
        # state - not to be confused with possible_actions(), which is the list of
        # moves available FROM it. Only the start state has action == None, because
        # it was created directly with HanoiState() rather than by a move, so this
        # check just skips the "Move disk..." line for the very first state.
        if state.action is not None:
            print(f"Move disk from peg {state.action[0]} to {state.action[1]}")
        print(state)
    #a "step" is a move between two states, so there's always one fewer move than there are states.
    print(f"Total {len(final_path)-1} steps")

depth_first_search()


Solution found!
Explored 81 states
Generated 242 states

 1 | |
 2 | |
 3 | |
 4 | |
 5 | |

Move disk from peg 0 to 2
 | | |
 2 | |
 3 | |
 4 | |
 5 | 1

Move disk from peg 0 to 1
 | | |
 | | |
 3 | |
 4 | |
 5 2 1

Move disk from peg 2 to 1
 | | |
 | | |
 3 | |
 4 1 |
 5 2 |

Move disk from peg 0 to 2
 | | |
 | | |
 | | |
 4 1 |
 5 2 3

Move disk from peg 1 to 2
 | | |
 | | |
 | | |
 4 | 1
 5 2 3

Move disk from peg 1 to 0
 | | |
 | | |
 2 | |
 4 | 1
 5 | 3

Move disk from peg 2 to 1
 | | |
 | | |
 2 | |
 4 | |
 5 1 3

Move disk from peg 0 to 2
 | | |
 | | |
 | | |
 4 | 2
 5 1 3

Move disk from peg 1 to 2
 | | |
 | | |
 | | 1
 4 | 2
 5 | 3

Move disk from peg 0 to 1
 | | |
 | | |
 | | 1
 | | 2
 5 4 3

Move disk from peg 2 to 1
 | | |
 | | |
 | | |
 | 1 2
 5 4 3

Move disk from peg 2 to 0
 | | |
 | | |
 | | |
 2 1 |
 5 4 3

Move disk from peg 1 to 2
 | | |
 | | |
 | | |
 2 | 1
 5 4 3

Move disk from peg 0 to 1
 | | |
 | | |
 | | |
 | 2 1
 5 4 3

Move disk from peg 2 to 1
 | | |
 | | |

Answer the following questions of your depth first search solution:
* Is the result of optimal length (in number of steps)? 
- no as we know from breadth-first it is possible to find the goal in 2^5 - 1 = 31 steps
* Does it generate more or fewer states, using the same metrics as the previous code? 
- DFS generated and explored fewer states than BFS. 
* When do you think we'd be likely to use depth first search, and when would we use breadth first?
- if memory is limited you could use depth-first, but if you are after the optimal solution and memory is not a concern, go with breadth-first search. 
- DFS is also a good option when you know there will be lots of solutions so you will find one quickly while BFS is a good option if there are fewer solutions or you need the shortest one. 
* If this version is not optimal, what modifications would be necessary to find an optimal result? 
- This version (DFS) isn't optimal as it explores branches as deep as they go and returns the first goal it gets even if not optimal. 
- To optimise this, first add a depth limit $l$ so the search never expands a state deeper than $l$ which is known as: depth-limited search. This only really works if you have a good idea of what $l$ should be. 
- Building on the above we can run Iterative deepening DFS (IDDFS) where you run a depth-limited search but with $l$ = 0, 1, 2 and so on until you find a solution. The goal you find will thus be optimal as all the shorter paths have already been searched fully. This is like BFS in that it explores the states in the same order and finds the optimal solution. The difference is that in IDDFS only the current path needs to be stored, provided wee don't keep the explored list and loops are avoided by checking the current path's parent chain instead. The cost of this is re-exploring shallow levels on each pass but this is only a small constant factor.

### Aside: depth first search as a recursive function

The exercise mentioned DFS "can also be implemented with a recursive function". Here is what that looks like. The idea: to search from a state, check whether it's the goal, and if not, search from each of its children in turn – where "search from" is the same function calling itself.

There is **no frontier list** at all. Python's call stack *is* the frontier: each pending `dfs(...)` call is a state whose remaining children haven't been tried yet, and returning from a call is backtracking. That is why recursion is the natural fit for depth first (and not for breadth first, which needs a queue).

Things to notice compared with the iterative `depth_first_search()` above:

* The `for` loop descends into a child **immediately**, before looking at the next sibling. The iterative version generates *all* siblings first, then pops one. Both are depth first, but they tie-break differently, so the path found and the counts are different (both non-optimal).
* `nonlocal generated:` dfs() lives inside recursive_depth_first_search(), and generated belongs to the outer function. An inner function can read outer variables freely, but the moment it tries to assign to one (generated += 1 is an assignment), Python assumes you mean a brand-new local variable and gets confused. `nonlocal` tells it "no, I mean the outer one". explored doesn't need it because explored.append(...) changes the list's contents without assigning to the name explored.
* Why the difference? Because integers are immutable — you can't change a 0 into a 1, you can only rebind the name to a different number. Lists are mutable — you can change the list's contents in place without touching the name. So a counter has to reassign (needs nonlocal); a list can be mutated (doesn't).
* Python limits recursion depth (about 1000 calls by default). A 5-disk DFS goes over 100 deep here, which is fine, but a bigger search space could crash with `RecursionError` – a practical reason the iterative version is usually preferred.

In [19]:
def recursive_depth_first_search(disks=5):
    explored = []
    generated = 0

    def dfs(state):
        nonlocal generated
        if state.is_goal_state():
            return state

        explored.append(state)
        for action in state.possible_actions():
            generated += 1
            new_state = state.next_state(action[0], action[1])
            if new_state not in explored:
                result = dfs(new_state)       # go deeper straight away
                if result is not None:
                    return result             # pass the goal back up the chain
        return None                           # dead end: backtrack to the caller

    goal = dfs(HanoiState(disks=disks))

    if goal is None:
        print("No solution found")
        return

    print("Solution found!")
    print(f"Explored {len(explored)} states")
    print(f"Generated {generated} states")
    print()

    final_path = []
    current_state = goal
    while current_state.parent is not None:
        final_path.append(current_state)
        current_state = current_state.parent
    final_path.append(current_state)

    for state in reversed(final_path):
        if state.action is not None:
            print(f"Move disk from peg {state.action[0]} to {state.action[1]}")
        print(state)
    print(f"Total {len(final_path)-1} steps")


recursive_depth_first_search()

Solution found!
Explored 162 states
Generated 309 states

 1 | |
 2 | |
 3 | |
 4 | |
 5 | |

Move disk from peg 0 to 1
 | | |
 2 | |
 3 | |
 4 | |
 5 1 |

Move disk from peg 0 to 2
 | | |
 | | |
 3 | |
 4 | |
 5 1 2

Move disk from peg 1 to 0
 | | |
 1 | |
 3 | |
 4 | |
 5 | 2

Move disk from peg 0 to 2
 | | |
 | | |
 3 | |
 4 | 1
 5 | 2

Move disk from peg 0 to 1
 | | |
 | | |
 | | |
 4 | 1
 5 3 2

Move disk from peg 2 to 0
 | | |
 | | |
 1 | |
 4 | |
 5 3 2

Move disk from peg 0 to 1
 | | |
 | | |
 | | |
 4 1 |
 5 3 2

Move disk from peg 2 to 0
 | | |
 | | |
 2 | |
 4 1 |
 5 3 |

Move disk from peg 1 to 0
 | | |
 1 | |
 2 | |
 4 | |
 5 3 |

Move disk from peg 0 to 2
 | | |
 | | |
 2 | |
 4 | |
 5 3 1

Move disk from peg 0 to 1
 | | |
 | | |
 | | |
 4 2 |
 5 3 1

Move disk from peg 2 to 0
 | | |
 | | |
 1 | |
 4 2 |
 5 3 |

Move disk from peg 0 to 1
 | | |
 | | |
 | 1 |
 4 2 |
 5 3 |

Move disk from peg 0 to 2
 | | |
 | | |
 | 1 |
 | 2 |
 5 3 4

Move disk from peg 1 to 0
 | | |
 | | 

### Notes: why iterative deepening is optimal but not "just BFS"

#### Memory: what each algorithm has to remember while running

**BFS** works level by level. Before it can look at any depth-10 state it must have generated *all* the depth-9 states and be holding them in the frontier. The number of states per level grows exponentially, so the frontier gets huge. That is BFS's weakness.

**DFS tree search** (the textbook version) only remembers the *path it is currently on* – start → child → grandchild → … → the state it is looking at right now – plus the siblings it hasn't tried yet. When it backtracks it forgets the branch it just abandoned. Memory grows with the *depth* of the path, not the *width* of the search space.

**Catch:** my `depth_first_search()` above keeps an `explored` list of every state ever visited and never removes anything, so it grows to the same size as it would in BFS. It is DFS in *order* (dives deep first) but does not get the memory benefit. To actually get low memory you would drop `explored` entirely and instead avoid infinite loops by checking whether a new state is already one of the current state's *ancestors* (walk up the `.parent` chain, same idea as building `final_path`).

#### Work: why "re-exploring shallow levels" only costs a small constant factor

Toy example – branching factor 2, goal at depth 3:

```
depth 0:              S
                    /   \
depth 1:           A     B
                  / \   / \
depth 2:         C   D E   F
                / \ ...
depth 3:       G   H  ...   <- goal is somewhere here
```

States per depth: **1, 2, 4, 8**.

Each IDDFS pass is a fresh DFS that refuses to go deeper than $l$:

| Pass | limit $l$ | What it does | States visited |
|---|---|---|---|
| 1 | 0 | Looks at S. Not the goal. Can't go deeper. Give up. | 1 |
| 2 | 1 | Start again at S. Visits S, A, B. Give up. | 1 + 2 = 3 |
| 3 | 2 | Start again at S. Visits S, A, B, C, D, E, F. Give up. | 1 + 2 + 4 = 7 |
| 4 | 3 | Start again at S. Visits everything down to depth 3. Finds the goal. | 1 + 2 + 4 + 8 = 15 |

**Total work: 1 + 3 + 7 + 15 = 26 states visited.** BFS visits each state once: **15**.

Why 26 not 15: S was visited in all 4 passes, A and B in 3, C–F in 2, the depth-3 states once. That is the "re-exploring shallow levels on each pass" – every time the limit is bumped and the search restarts, it walks through the same shallow states again on the way down.

But notice where the 26 came from: the **final pass alone was 15**. The three "wasted" passes only added 11 between them, because the shallow levels are small. The deeper the goal, the more the last pass dominates and the smaller the repeats are as a fraction of the total. So IDDFS ≈ (BFS work) × (a fixed multiplier, e.g. ~1.1 for branching factor 10), not anything exponential.

#### Summary

- **BFS:** remembers everything, does the work once.
- **IDDFS:** remembers almost nothing (just the current path), does the shallow work repeatedly – but the repeated part is cheap.
- Both explore states in the same depth order, so both are optimal for the same reason: no depth-$n$ solution is accepted until every depth-$(n-1)$ possibility has been ruled out.


- BFS: remembers everything, does the work once.
- IDDFS: remembers almost nothing, does the shallow work repeatedly — but the repeated part is cheap.

### Iterative deepening depth first search (IDDFS)

Below is an implementation of the fix described in Q4. Note the differences from `depth_first_search()` above:

* **No `explored` list.** This is depth first *tree* search, so memory is just the frontier (the current path plus untried siblings). Instead, `on_current_path()` stops us going round in circles by checking whether a new state is already one of the current state's ancestors (walking up the `.parent` chain).
* **A depth limit.** `depth_limited_search(limit)` refuses to expand any state whose depth is `>= limit`, so it can never find a path longer than `limit`.
* **An outer loop** in `iterative_deepening_search()` that calls it with `limit = 0, 1, 2, ...` until a solution appears. Because every shorter limit has already been searched exhaustively, the first solution found is optimal.

Watch the per-pass numbers it prints: the final pass does most of the work, and the repeated shallow passes are a small fraction of the total - exactly the "small constant factor" from the notes above.

**Use 3 or 4 disks, not 5.** With no explored list, the search explores every distinct *path* rather than every distinct *state*. Hanoi's state graph is tiny (3⁵ = 243 states) but has many routes between the same states, so the number of paths of length 31 is astronomical - the last few passes with 5 disks take minutes each. With 3 disks the goal is at depth 7 and it runs instantly.

#### The big picture: two functions, two jobs

**`depth_limited_search(limit)` – the worker.** Given a number, it does a DFS but refuses to go more than `limit` moves deep. It either finds the goal within that many moves (returns the goal state) or doesn't (returns `None`). All the pops, pushes and `continue`s live in here.

**`iterative_deepening_search()` – the boss.** It knows nothing about frontiers. All it does is:

> "Worker, try with limit 0." – *Nothing.*
> "Try with limit 1." – *Nothing.*
> "Try with limit 2." – *Nothing.*
> …
> "Try with limit 7." – *Found it!*
> "Great, print the path."

The worker does one bounded search; the boss calls it repeatedly with a bigger number each time. Note that `limit` is a *parameter* of the worker – the worker never chooses it, the boss passes it in.

**Why it's optimal:** the boss tried 0 through 6 and the worker came back empty every time, so there is no solution shorter than 7. The first one found must be the shortest.

**Reading order vs running order:** the worker is written above the boss because Python must see a function's definition before it can be called – but nothing *runs* until the last line, `iterative_deepening_search(disks=3)`. To follow the execution, start at the bottom and jump up.

Everything else in the code – tuples, unpacking, `pop()`, `continue`, `on_current_path()` – is just the plumbing that lets the worker do its one job.

In [ ]:
def on_current_path(state, new_state):
    # is new_state already one of state's ancestors (or state itself)?
    # this replaces the explored list: it only stops us going round in circles
    # along the path we are currently on, and costs nothing extra to store
    while state is not None:
        if state == new_state:
            return True
        state = state.parent
    return False


def depth_limited_search(limit, disks=3):
    # depth first TREE search: no explored list, and never expand deeper than limit
    # note this is TREE search since we do not remember states we already visited via explored.  
    state = HanoiState(disks=disks)
    # the frontier stores (state, depth) pairs, so we always know how deep a
    # state is without having to walk back up its parent chain to count
    frontier = [(state, 0)]
    generated = 0
    # Expanding" a state means taking it off the frontier, generating all its children 
    # (calling possible_actions() and next_state() for each), and adding them to the frontier.
    expanded = 0

    while len(frontier) > 0:
        current_state, current_depth = frontier.pop()
        if current_state.is_goal_state():
            return current_state, expanded, generated

        if current_depth >= limit:
            # continue means "skip the rest of this loop body and go straight back to the top for the next iteration"
            # The while loop just goes round again and pops the next thing off the frontier
            # Why >=?
            # A state at depth limit has already used up all its moves, 
            # so its children would be at depth limit + 1 which is too deep.
            continue  # at the limit: don't expand, just backtrack

        expanded += 1
        # loop over every legal move from here:
        for action in current_state.possible_actions():
            # generated += 1 — count every child we build, whether or not it gets kept:
            generated += 1
            new_state = current_state.next_state(action[0], action[1])
            #  keep it unless it's a repeat of somewhere on the route we took to get here:
            if not on_current_path(current_state, new_state):
                frontier.append((new_state, current_depth + 1))

    return None, expanded, generated  # nothing found within this limit


def iterative_deepening_search(disks=3):
    total_expanded = 0
    total_generated = 0
    limit = 0
    while True:
        result, expanded, generated = depth_limited_search(limit, disks)
        total_expanded += expanded
        total_generated += generated
        print(f"limit {limit:2d}: expanded {expanded:5d}, generated {generated:5d}"
              + ("  <- solution found" if result is not None else ""))
        if result is not None:
            break
        limit += 1

    print()
    print("Solution found!")
    print(f"Expanded {total_expanded} states in total across all passes")
    print(f"Generated {total_generated} states in total across all passes")
    print()

    # same backwards walk as before: goal -> start, then print reversed
    final_path = []
    current_state = result
    while current_state.parent is not None:
        final_path.append(current_state)
        current_state = current_state.parent
    final_path.append(current_state)

    for state in reversed(final_path):
        if state.action is not None:
            print(f"Move disk from peg {state.action[0]} to {state.action[1]}")
        print(state)
    print(f"Total {len(final_path)-1} steps")


iterative_deepening_search(disks=3)

# depth-first search part (the worker)

```python
while len(frontier) > 0:          ← top of the loop
    pop                           ← one state comes off (pop() = newest = deepest)
    goal check                    ← if goal: return it immediately, loop stops here
    depth check (maybe continue)
    expand: for loop pushes children
                                  ← bottom; go back to top
```

One call to the worker with `limit = 2` (3 disks):

- Iteration 1: pop start (d0), push A and B.
- Iteration 2: pop B (d1), push B1.
- Iteration 3: pop B1 (d2), hit the limit, continue.
- Iteration 4: pop A (d1), push A1.
- Iteration 5: pop A1 (d2), hit the limit, continue.
- Frontier empty → while condition false → loop ends → return None.

Two ways out: goal found (return the state mid-loop) or frontier empty (return None).

If you are stuck, you can [click here](additional/depth_first.ipynb) to see a solution to the depth first search problem.